# Coding entire Transformer Block

In [2]:
import torch.nn as nn
import torch

In [3]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, # vocabulary size
    "context_length": 1024, # context length
    "emb_dim": 768, # embedding dimension
    "n_heads": 12,  # no. of attention heads
    "n_layers": 12, # no. of transformer layers
    "drop_rate": 0.1, # dropout rate
    "qkv_bias": False # query-key-value bias
}

## MultiHead Attention

In [9]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()

    assert (d_out % num_heads) == 0, "d_out must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    # calculate individual head dimension according to d_out and no. of heads present
    self.head_dim = d_out // num_heads

    # random key,query,value initialization with d_in and d_out)
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.out_proj = nn.Linear(d_in, d_out) # Linear layer to combine head outputs
    self.dropout = nn.Dropout(dropout)

    self.register_buffer(
        "mask",
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x):
    b, num_tokens, d_in = x.shape # initialize (Batch, token_size, input_dimension)

    # keys, values queries (random of d_in,d_out(dimensions) multiplied with inputs)
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    # convert of each head i.e d_out --> num_heads and head_dimension
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Group matrices by num_heads for parallel computation.

    #(b,num_tokens,num_heads,head_dim) --> (b, num_heads, num_tokens, head_dim)
    # (1,3,2,3) --> (1,2,3,3) (The positions 1 and 2 will be transposed)
    keys = keys.transpose(1,2)
    queries = queries.transpose(1,2)
    values = values.transpose(1,2)

    # now for each query we will do matmul with keys.
    # and for that we need to transpose the postion 2 and 3 of keys.
    # (b,num_heads,num_tokens,head_dim) * (b, num_heads, head_dim, num_tokens)
    #                                    |
    #                    (b,num_heads,num_tokens,num_tokens)
    attn_scores = queries @ keys.transpose(2,3)

    # masking
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

    attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)

    # softmax with Sqrt of head_dim and dropout
    attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    # calulate context vector with d_out as dimension preserved

    # (b,num_heads,num_tokens,num_tokens) * (b,num_heads,num_tokens,head_dim)
    #                                     |
    #                     (b,num_heads,num_tokens,head_dim)
    #                                     | (1,2) transpose
    #                     (b,num_tokens,num_heads,head_dim)
    context_vector = (attn_weights @ values).transpose(1,2)
    # now we can merge num_heads and head_dim easily to d_out.
    # we merge the num_heads and head_dim into single row giving d_out dimension.
    # (b,num_tokens,num_heads,head_dim) --> (b,num_tokens,d_out)
    # contiguous ensures that after reshaping the values stay in same block of memory.
    context_vector = context_vector.contiguous().view(b, num_tokens, self.d_out)
    context_vector = self.out_proj(context_vector)

    return context_vector

## Layer normalization, GeLU and Feed-Forward neural network

In [10]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    # if unbiased is 'True', it applied Bessels correction which is divide by n-1 for variance not by n.
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + self.eps) # eps --> Epsilon is used to prevent division by 0 during normalization.
    return self.scale * norm_x + self.shift # scale and shifts are trainable parameters used to tweak norms.

In [11]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2 / torch.pi)) *
        (x + 0.044715 * torch.pow(x, 3))
    ))

In [12]:
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), # Expansion
        GELU(), # Activation
        nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), # Contraction
    )

  def forward(self, x):
    return self.layers(x)

## Transformer Block

In [14]:
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att = MultiHeadAttention(
        d_in = cfg["emb_dim"],
        d_out = cfg["emb_dim"],
        context_length = cfg["context_length"],
        num_heads = cfg["n_heads"],
        dropout = cfg["drop_rate"],
        qkv_bias = cfg["qkv_bias"]
    )
    self.ff = FeedForward(cfg)
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])
    self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # shortcut connection for attention block
    shortcut = x
    x = self.norm1(x) # normalization
    x = self.att(x) # attention
    x = self.drop_shortcut(x) # dropout
    x = x + shortcut # add the original input back

    # shortcut connection for feed forward block
    shortcut = x
    x = self.norm2(x) # normalization
    x = self.ff(x) # feed forward
    x = self.drop_shortcut(x) # dropout
    x = x + shortcut # add the original input back

    return x

In [15]:
torch.manual_seed(123)
x = torch.rand(2,4,768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape: ", x.shape)
print("Output shape: ", output.shape)

Input shape:  torch.Size([2, 4, 768])
Output shape:  torch.Size([2, 4, 768])
